# Odds Poisson Model\n\nNutzt Wettquoten (Marktdurchschnitt `AvgH/AvgD/AvgA` von football-data.co.uk) statt Team-Identität. Die Kombination aus Team-Einhot-Spalten und Quoten wurde ausprobiert, aber verworfen - zu kollinear (Quoten preisen Team-Stärke schon ein), die reine Quoten-Version schnitt beim Test auf 25/26 besser ab. Auch der Heimvorteil wird nicht separat geschätzt, sondern ist implizit schon in den Quoten enthalten.\n\nDaten: `data/raw/odds/` (andere Quelle als die übrigen Modelle - `football-data.co.uk` statt `datahub.io`, dieselben Spiele, aber mit Quoten-Spalten).\n\nFür den Modellvergleich siehe `model_comparison.ipynb`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.csv_source import CSVSource
from src.data.loader import DataLoader
from src.models.odds_poisson.features import build_design_matrix, implied_probabilities
from src.models.odds_poisson.predict import predict_score
from src.models.poisson_regressor import PoissonRegressor
from src.models.scoring import kicktipp_points

In [ ]:
DATA_DIR = Path.cwd().parent / "data" / "raw" / "odds"
TRAIN_SEASONS = ["season-2122.csv", "season-2223.csv", "season-2324.csv", "season-2425.csv"]

frames = []
for filename in TRAIN_SEASONS:
    loader = DataLoader(CSVSource(DATA_DIR / filename))
    frames.append(loader.load())

train_df = pd.concat(frames, ignore_index=True)
train_df.shape

In [ ]:
X, y, team_index = build_design_matrix(train_df)
X.shape, y.shape

In [ ]:
model = PoissonRegressor(learning_rate=0.1, n_iterations=5000)
model.fit(X, y)

In [ ]:
plt.plot(model.loss_history_)
plt.xlabel("Iteration")
plt.ylabel("Loss (mean negative log-likelihood)")
plt.title("Training loss")
plt.show()

print("Koeffizienten [is_home, own_win_prob, opponent_win_prob, draw_prob]:")
print(model.coef_)

## Vergleich mit Saison 25/26\n\nQuoten sind pro Spiel schon vorhanden (kein Formkurve-Update nötig, im Gegensatz zu den form-basierten Modellen) - jedes Spiel nutzt direkt seine eigenen Vorab-Quoten.

In [ ]:
test_df = DataLoader(CSVSource(DATA_DIR / "season-2526.csv")).load()
test_df["date"] = pd.to_datetime(test_df["date"], format="%d/%m/%Y")
test_df = test_df.sort_values("date", kind="stable").reset_index(drop=True)
test_df["matchday"] = test_df.index // 9 + 1
test_df.shape

In [ ]:
results = []
for match in test_df.itertuples():
    predicted = predict_score(model, match.avgh, match.avgd, match.avga)
    actual = (int(match.fthg), int(match.ftag))
    results.append({
        "matchday": match.matchday,
        "home": match.hometeam,
        "away": match.awayteam,
        "predicted": f"{predicted[0]}:{predicted[1]}",
        "actual": f"{actual[0]}:{actual[1]}",
        "points": kicktipp_points(predicted, actual),
    })

results_df = pd.DataFrame(results)
results_df.head(9)

In [ ]:
points_per_matchday = results_df.groupby("matchday")["points"].sum()

plt.plot(points_per_matchday.index, points_per_matchday.values, marker="o")
plt.xlabel("Spieltag")
plt.ylabel("Punkte")
plt.title("Kicktipp-Punkte pro Spieltag (Saison 25/26, Quoten-Modell)")
plt.show()

print(f"Gesamtpunkte: {results_df['points'].sum()} ({results_df['points'].mean():.2f} im Schnitt pro Spiel)")
print("Verteilung:")
print(results_df["points"].value_counts().sort_index(ascending=False))

In [ ]:
def categorize(points: int) -> str:
    if points == 4:
        return "exakt"
    if points in (2, 3):
        return "tendenz"
    return "falsch"

results_df["category"] = results_df["points"].apply(categorize)

matchday_summary = (
    results_df
    .groupby(["matchday", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["exakt", "tendenz", "falsch"], fill_value=0)
)
matchday_summary